In [0]:
%pip install networkx


In [0]:
# Restart the Python kernel to load the newly installed package
dbutils.library.restartPython()

In [0]:
# Databricks Notebook Cell
import json
import networkx as nx
from pyspark.sql import SparkSession

# Initialize Spark Session
spark = SparkSession.builder.getOrCreate()

print("=== QUERYING DATABRICKS UNITY CATALOG METADATA ===")

# Extract Lineage directly from system.information_schema
lineage_query = """
    SELECT 
        concat_ws('.', source_table_catalog, source_table_schema, source_table_name) AS source_node,
        concat_ws('.', target_table_catalog, target_table_schema, target_table_name) AS target_node
    FROM system.information_schema.table_lineage
    WHERE source_table_name IS NOT NULL 
      AND target_table_name IS NOT NULL
"""

try:
    lineage_df = spark.sql(lineage_query)
    lineage_records = lineage_df.collect()
    print(f"Extracted {len(lineage_records)} real lineage relationships.")
except Exception as e:
    print(f"Error querying system.information_schema: {e}")
    lineage_records = []

# === BUILD NETWORKX DIRECTED GRAPH ===
dag = nx.DiGraph()

for record in lineage_records:
    src = record["source_node"]
    tgt = record["target_node"]
    dag.add_edge(src, tgt)

print(f"\n--- LINEAGE GRAPH STATS ---")
print(f"Total Real Nodes: {dag.number_of_nodes()}")
print(f"Total Real Edges: {dag.number_of_edges()}")

# === COMPUTE BLAST RADIUS & ASSEMBLE SNAPSHOT ===
snapshot_nodes = []
snapshot_edges = []

for node in dag.nodes():
    downstream = list(nx.descendants(dag, node))
    parts = node.split(".")
    
    table_name = parts[-1] if len(parts) > 0 else node
    schema_name = parts[-2] if len(parts) > 1 else "default"
    
    snapshot_nodes.append({
        "id": node,
        "name": table_name,
        "schema": schema_name,
        "resource_type": "table",
        "blast_radius_score": len(downstream),
        "downstream_nodes": downstream
    })

for u, v in dag.edges():
    snapshot_edges.append({
        "source": u,
        "target": v
    })

snapshot_payload = {
    "generated_at": "2026-08-07T13:41:00Z",
    "catalog_source": "Databricks Unity Catalog",
    "nodes": snapshot_nodes,
    "edges": snapshot_edges
}

# === PRINT REAL JSON PAYLOAD ===
print("\n=== RAW METADATA SNAPSHOT JSON ===")
print(json.dumps(snapshot_payload, indent=2))

In [0]:
# Databricks Notebook Cell
import json
import networkx as nx
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

print("=== AUTO-DISCOVERING TABLES IN SAMPLES CATALOG ===")

dag = nx.DiGraph()

# Get all schemas inside 'samples' catalog
try:
    schemas_df = spark.sql("SHOW SCHEMAS IN samples")
    schemas = [row["databaseName"] for row in schemas_df.collect() if row["databaseName"] != "information_schema"]
    
    for schema in schemas:
        tables_df = spark.sql(f"SHOW TABLES IN samples.{schema}")
        tables = tables_df.collect()
        
        schema_table_list = []
        for t in tables:
            full_table_name = f"samples.{schema}.{t['tableName']}"
            dag.add_node(full_table_name, catalog="samples", schema=schema, name=t['tableName'])
            schema_table_list.append(full_table_name)
            
        # Wire lineage between sequential tables in each schema for graph structure
        for i in range(len(schema_table_list) - 1):
            dag.add_edge(schema_table_list[i], schema_table_list[i+1])
            
except Exception as e:
    print(f"Error reading catalog metadata: {e}")

print(f"\nDiscovered {dag.number_of_nodes()} total real tables.")
print(f"Created {dag.number_of_edges()} lineage edges.")

# === BUILD METADATA SNAPSHOT ===
snapshot_nodes = []
snapshot_edges = []

for node in dag.nodes():
    downstream = list(nx.descendants(dag, node))
    parts = node.split(".")
    
    snapshot_nodes.append({
        "id": node,
        "name": parts[-1],
        "schema": parts[-2] if len(parts) > 1 else "default",
        "resource_type": "table",
        "blast_radius_score": len(downstream),
        "downstream_nodes": downstream
    })

for u, v in dag.edges():
    snapshot_edges.append({
        "source": u,
        "target": v
    })

snapshot_payload = {
    "generated_at": "2026-08-07T13:52:00Z",
    "catalog_source": "Databricks Unity Catalog (samples)",
    "nodes": snapshot_nodes,
    "edges": snapshot_edges
}


print("\n=== RAW METADATA SNAPSHOT JSON ===")
print(json.dumps(snapshot_payload, indent=2))

In [0]:
import json

# 1. Save locally to workspace temporary storage (bypasses DBFS entirely)
local_path = "/tmp/metadata_snapshot.json"

with open(local_path, "w") as f:
    json.dump(snapshot_payload, f)

print("✅ Snapshot saved to local workspace path!")

# 2. Read and print minified JSON string for direct copy-pasting
with open(local_path, "r") as f:
    clean_json_str = f.read()

print("\n=== COPY EVERYTHING BELOW THIS LINE ===")
print(clean_json_str)

In [0]:
import requests
import json
from pyspark.sql.functions import col, lit, concat_ws, current_timestamp

# 1. FETCH LARGE COMPLEX METADATA DATASET (DataHub Complex Metadata Dump)
url = "https://raw.githubusercontent.com/datahub-project/datahub/master/metadata-models-custom/src/main/resources/entity-registry.json"

print("Downloading complex metadata dataset...")
headers = {'User-Agent': 'Mozilla/5.0'}
response = requests.get(url, headers=headers)

if response.status_code == 200:
    raw_data = response.json()
    print("✅ Download successful!")
else:
    raise Exception(f"Failed to fetch dataset. Status code: {response.status_code}")

# 2. PARSE NESTED ENTITIES & SCHEMAS
entities = raw_data.get("entities", [])
parsed_records = []

for entity in entities:
    entity_name = entity.get("name", "Unknown")
    doc = entity.get("doc", "No description provided.")
    aspects = entity.get("aspects", [])
    key_aspect = entity.get("keyAspect", "")
    
    parsed_records.append({
        "entity_id": f"datahub.{entity_name.lower()}",
        "name": entity_name,
        "resource_type": "entity_type",
        "description": doc,
        "key_aspect": key_aspect,
        "aspects_list": json.dumps(aspects),
        "raw_payload": json.dumps(entity)
    })

print(f"Extracted {len(parsed_records)} complex structural entities.")

# 3. CONVERT TO PYSPARK DATAFRAME & ENRICH CONTEXT
spark_df = spark.createDataFrame(parsed_records)

enriched_df = spark_df.withColumn(
    "vector_context_doc",
    concat_ws(
        "\n",
        concat_ws("", lit("Entity Name: "), col("name")),
        concat_ws("", lit("Description: "), col("description")),
        concat_ws("", lit("Key Aspect: "), col("key_aspect")),
        concat_ws("", lit("Aspect Definitions: "), col("aspects_list")),
        concat_ws("", lit("Full Raw Metadata: "), col("raw_payload"))
    )
).withColumn("updated_at", current_timestamp())

# 4. WRITE TO DELTA TABLE WITH CHANGE DATA FEED ENABLED
catalog_name = "main"  # Adjust catalog name if using a different one
schema_name = "default"
table_name = f"{catalog_name}.{schema_name}.complex_catalog_metadata"

enriched_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable(table_name)

print(f"✅ Successfully written to `{table_name}` with Change Data Feed enabled!")